In [1]:
%reload_ext autoreload
%autoreload 2

import pandas as pd
import altair as alt
from preprocessing import create_clean_dataframe
from basic_eda import *
import numpy as np

# Import Data

In [2]:
path = "../data/simpsons_episodes.csv"

# Data Preprocessing

In [3]:
df = create_clean_dataframe(path)

# Data Quality checks

## Null values
The dataset contains a small number of missing values. Specifically, `imdb_rating` has 3 missing entries, `us_viewers_in_millions` has 6 missing values, and `views` has 4 missing values. All other columns are complete with no null values.


In [4]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   season                  600 non-null    int64         
 1   number_in_season        600 non-null    int64         
 2   original_air_date       600 non-null    datetime64[ns]
 3   day_aired               600 non-null    object        
 4   imdb_rating             597 non-null    float64       
 5   us_viewers_in_millions  594 non-null    float64       
 6   views                   596 non-null    float64       
 7   title                   600 non-null    object        
 8   is_last_episode         600 non-null    int64         
dtypes: datetime64[ns](1), float64(3), int64(3), object(2)
memory usage: 42.3+ KB
None


## Unique values
An inspection of the unique values reveals some interesting patterns in the dataset. There are 28 unique seasons and the maximum episode number within a season is 25, which is consistent with typical TV season structures. The `original_air_date` column contains 591 unique dates out of 600 episodes, meaning that 9 episodes share the same air date. The `day_aired` variable has only 5 unique values and is highly unbalanced: most episodes were aired on Sunday (506), followed by Thursday (89), while Tuesday and Wednesday only appear twice each and Friday only once. This strong imbalance suggests that the show was predominantly broadcast on a single day of the week.


In [5]:
print_unique_values(df)
df["day_aired"].value_counts()


Unique values per column:
season                     28
number_in_season           25
original_air_date         591
day_aired                   5
imdb_rating                39
us_viewers_in_millions    380
views                     590
title                     600
is_last_episode             2
dtype: int64


day_aired
Sunday       506
Thursday      89
Tuesday        2
Wednesday      2
Friday         1
Name: count, dtype: int64

## Unique values 2

no té sentit mirar als uniques de imdb_rating, views, is_last_episode us_viewers_in_millions

Contem el numero de capitols per temporada

In [6]:
df.groupby("season").size().reset_index(name="episode_count").sort_values("season")

,season,episode_count
0,1,13
1,2,22
2,3,24
3,4,22
4,5,22
5,6,25
6,7,25
7,8,25
8,9,25
9,10,23


## Outlier detection
An outlier detection analysis was performed using the IQR method on the numerical variables `imdb_rating`, `us_viewers_in_millions`, and `views`. The results show a very small number of outliers in `imdb_rating` (2) and a slightly higher number in `us_viewers_in_millions` (9). In contrast, the `views` variable contains a larger number of potential outliers (41), suggesting a higher variability in this metric. These values should be considered during the analysis, as extreme observations could influence some visualizations or statistical interpretations.

In [7]:
columns_to_check = ["imdb_rating", "us_viewers_in_millions", "views"]
print_outliers(df, columns_to_check)


Number of outliers (IQR method):
imdb_rating: 2
us_viewers_in_millions: 9
views: 41


## Outlier Detection 2

jo em petava la variable de views així ens estalviem una. A part no és interessant

### IMDB Rating

In [8]:
outliers_imdb_rating = count_outliers_iqr(df, "imdb_rating")
lb = outliers_imdb_rating[2]
ub = outliers_imdb_rating[3]
outliers_imdb_rating[0]

,season,number_in_season,original_air_date,day_aired,imdb_rating,us_viewers_in_millions,views,title,is_last_episode
71,9,11,1998-01-04,Sunday,5.1,8.90,15780.0,"All Singing, All Dancing",0
193,23,22,2012-05-20,Sunday,4.5,4.82,44434.0,Lisa Goes Gaga,1


We find 2 outliers belonging to very low ratings. This are:
* The 11th episode of season 9
* The 22th episode of season 11

Since we don't have many of them and the range of values 0-10 of the variable is not very large, keeping them won't affect visualizations and we decide to do so

In [9]:
base = alt.Chart(df).transform_calculate(
    outlier=f"(datum.imdb_rating < {lb}) || (datum.imdb_rating > {ub})"
)

hist = base.mark_bar().encode(
    x=alt.X("imdb_rating:Q", bin=alt.Bin(maxbins=30), title="IMDb rating"),
    y=alt.Y("count():Q", title="Episodes"),
    color=alt.condition("datum.outlier", alt.value("red"), alt.value("steelblue"))
)

bounds_df = pd.DataFrame({"bound": [lb, ub]})
bounds = alt.Chart(bounds_df).mark_rule(color="black", strokeDash=[4, 4]).encode(
    x="bound:Q"
 )

chart = (hist + bounds).properties(
    title="Distribution of IMDb Ratings with Outliers Highlighted"
 )
chart

alt.LayerChart(...)

# Us Viewers

We find 9 outliers on the variable us_viewers_in_millions, all of them belonging to early seasons of the show. Some sepcial events happened on these dates which may have affected the show's audience:
* On March 25, 1990, a tragic arson attack at the Happy Land social club in the Bronx, New York City, killed 87 people
* On April 29, 1990, the US space shuttle Discovery (STS-31) returned to Earth after deploying the Hubble Space Telescope. Major events included the LA Lakers defeating the Houston Rockets to set a record, Nirvana playing a legendary show in Washington, D.C., and Dan Quisenberry announcing his MLB retirement. 
* https://www.onthisday.com/date/1990/october/11
* The world's very first text message wasn't a love note, a meme, or a status update—it was a simple holiday greeting. On December 3, 1992, British engineer Neil Papworth sent the message “Merry Christmas” from his computer to the mobile phone of Vodafone director Richard Jarvis.

In [10]:
outliers_viewers = count_outliers_iqr(df, "us_viewers_in_millions")
lb = outliers_viewers[2]
ub = outliers_viewers[3]
outliers_viewers_df = outliers_viewers[0]
outliers_viewers_df['season'].value_counts()

season
1    6
2    2
4    1
Name: count, dtype: int64

In [11]:
# Ensure IQR bounds for viewers exist
if "lb" not in globals() or "ub" not in globals():
    outliers_viewers = count_outliers_iqr(df, "us_viewers_in_millions")
    lb = outliers_viewers[2]
    ub = outliers_viewers[3]

first_seasons = df[df["season"] < 5].copy()
first_seasons = first_seasons.sort_values(["season", "original_air_date"])

# Connect lines by removing null viewer values
line = (
    alt.Chart(first_seasons)
    .transform_filter("isValid(datum.us_viewers_in_millions)")
    .mark_line()
    .encode(
        x=alt.X("original_air_date:T", title="Original Air Date"),
        y=alt.Y("us_viewers_in_millions:Q", title="US Viewers (Millions)"),
        color=alt.Color("season:N", title="Season")
    )
)

# Vertical separators at the last episode of each season
season_separators = alt.Chart(
    first_seasons[first_seasons["is_last_episode"] == 1]
).mark_rule(color="black", strokeDash=[4, 4], opacity=0.6).encode(
    x=alt.X("original_air_date:T")
)

# Horizontal lower/upper bounds
bounds_df = pd.DataFrame(
    {
        "bound": [lb, ub],
        "label": ["Lower bound", "Upper bound"]
    }
)
bounds = alt.Chart(bounds_df).mark_rule(color="firebrick", strokeDash=[6, 4]).encode(
    y=alt.Y("bound:Q"),
    detail="label:N",
    tooltip=["label:N", alt.Tooltip("bound:Q", format=".2f")]
 )

chart = (line + season_separators + bounds).properties(
    title="US Viewers Distribution for First Seasons with Season Separators and IQR Bounds"
 )

chart

alt.LayerChart(...)

In [12]:
# density plot for viewers in all seasons with lines indifcating lower and upper bounds for outliers
chart = alt.Chart(df).transform_density(
    "us_viewers_in_millions",
    as_=["us_viewers_in_millions", "density"]
).mark_area(opacity=0.5, color="steelblue").encode(
    x=alt.X("us_viewers_in_millions:Q", title="US Viewers (Millions)"),
    y=alt.Y("density:Q", title="Density")
)

bounds = alt.Chart(bounds_df).mark_rule(color="firebrick", strokeDash=[6, 4]).encode(
    x=alt.X("bound:Q"),
    detail="label:N",
    tooltip=["label:N", alt.Tooltip("bound:Q", format=".2f")]
)

(chart + bounds)

alt.LayerChart(...)

In [13]:
alt.Chart(df).mark_bar().encode(
    x='day_aired',
    y='average(imdb_rating)'
)

alt.Chart(...)

The outliers are coaused of the decrease in the show's popularity. We'll keep them in the distribution

In [14]:
alt.Chart(df).mark_bar().encode(
    x='day_aired',
    y='count()'
)

alt.Chart(...)

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   season                  600 non-null    int64         
 1   number_in_season        600 non-null    int64         
 2   original_air_date       600 non-null    datetime64[ns]
 3   day_aired               600 non-null    object        
 4   imdb_rating             597 non-null    float64       
 5   us_viewers_in_millions  594 non-null    float64       
 6   views                   596 non-null    float64       
 7   title                   600 non-null    object        
 8   is_last_episode         600 non-null    int64         
dtypes: datetime64[ns](1), float64(3), int64(3), object(2)
memory usage: 42.3+ KB


## imdb_rating

The missing ratings from this variable come from the last season of the dataset, where there are only 4 episodes and 3 present in the data.

In [16]:
alt.Chart(df).mark_bar().encode(
    x='season',
    y='count()'
)

alt.Chart(...)

# us_viewers_in_millions

3 of the 6 missing values of this variable come from the espisodes of season 28, The other 3 come from episodes of season 8

In [17]:
df[df['us_viewers_in_millions'].isna()]

,season,number_in_season,original_air_date,day_aired,imdb_rating,us_viewers_in_millions,views,title,is_last_episode
59,8,7,1996-12-15,Sunday,7.8,NaN,60912.0,Lisa's Date with Density,0
65,8,20,1997-04-13,Sunday,7.7,NaN,54155.0,The Canine Mutiny,0
234,28,2,2016-10-02,Sunday,NaN,NaN,NaN,"Friends and Family""[203]",0
235,28,3,2016-10-09,Sunday,NaN,NaN,NaN,"The Town""[205]",0
236,28,4,2016-10-16,Sunday,NaN,NaN,NaN,"Treehouse of Horror XXVII""[207]",1
320,8,8,1996-12-29,Sunday,8.8,NaN,66281.0,Hurricane Neddy,0


# Decision

We detected that there are null values, so we found another data source that had the ratings of the three missing episodes.aving information of 4 more points won't help us to identifying trends any better compared to all the seasons we have more episodes of. We decide to eliminate epsiodes from season 28 and still have data of the seasons 1-27, which will be enough to answer the questions

In order to not have 3 empyt points for season 8 and make the visualkization akward, we can use other sources to fins the audience values for the missing episoed and plug them into our dataset.

In [18]:
[df['season'] == 8]

[0      False
 1      False
 2      False
 3      False
 4      False
        ...  
 595    False
 596    False
 597    False
 598    False
 599    False
 Name: season, Length: 600, dtype: bool]

we will use another data source to plug in the missing values for the episodes in season 8
https://www.kaggle.com/datasets/jonbown/simpsons-episodes-2016?select=simpsons_episodes.csv

In [19]:
dataset2 = pd.read_csv('../data/simpsons_episodes_second_source.csv')
dataset2[dataset2['season'] == 8]

,id,title,description,original_air_date,production_code,directed_by,written_by,season,number_in_season,number_in_series,us_viewers_in_millions,imdb_rating,tmdb_rating,tmdb_vote_count
153,153,Treehouse of Horror VII,Bart's Siamese twin swells in the attic; Lisa'...,1996-10-27,4F02,Mike B. Anderson,Ken Keeler,8,1,154,18.30,8.4,7.5,23
154,154,You Only Move Twice,"After moving to another town for a new job, Ho...",1996-11-03,3F23,Mike B. Anderson,John Swartzwelder,8,2,155,13.90,9.2,8.2,31
155,155,The Homer They Fall,Homer becomes a boxer after Moe discovers his ...,1996-11-10,4F03,Mark Kirkland,Jonathan Collier,8,3,156,17.00,8.1,7.4,28
156,156,"Burns, Baby Burns",The Simpsons try to reunite Mr. Burns with Lar...,1996-11-17,4F05,Jim Reardon,Ian Maxtone-Graham,8,4,157,12.60,7.6,6.5,24
157,157,Bart After Dark,Marge leads a crusade against a local burlesqu...,1996-11-24,4F06,Dominic Polcino,Richard Appel,8,5,158,14.10,8.3,7.6,24
158,158,A Milhouse Divided,Homer questions the state of his marriage afte...,1996-12-01,4F04,Steven Dean Moore,Steve Tompkins,8,6,159,12.80,8.0,7.3,23
159,159,Lisa's Date with Density,"Lisa develops a crush on Nelson, and tries to ...",1996-12-15,4F01,Susie Dietter,Mike Scully,8,7,160,12.20,7.8,7.0,22
160,160,Hurricane Neddy,After a hurricane destroys Ned Flanders' house...,1996-12-29,4F07,Bob Anderson,Steve Young,8,8,161,14.40,8.7,7.8,25
161,161,El Viaje Misterioso de Nuestro Jomer (The Myst...,After eating insanity peppers at a chili cook-...,1997-01-05,3F24,Jim Reardon,Ken Keeler,8,9,162,14.85,8.6,7.9,23
162,162,The Springfield Files,Special Agents Mulder and Scully arrive in Spr...,1997-01-12,3F25,Steven Dean Moore,Reid Harrison,8,10,163,20.41,9.0,8.4,26


In [20]:
df2 = df.merge(dataset2[['season', 'number_in_season', 'us_viewers_in_millions']], on=['season', 'number_in_season'], how='left', suffixes=('', '_from_ds2'))
df2['us_viewers_in_millions'] = df2['us_viewers_in_millions'].fillna(df2['us_viewers_in_millions_from_ds2'])
df2.drop(columns=['us_viewers_in_millions_from_ds2'], inplace=True)

In [21]:
df2.to_csv('../data/simpsons_episodes_cleaned.csv', index=False)

# Q1 How have the ratings evolved over time?

In [23]:
line_chart = alt.Chart(df2).mark_line().encode(
    x=alt.X('original_air_date:T', title='Original Air Date'),
    y=alt.Y('us_viewers_in_millions:Q', title='US Viewers (Millions)')
).properties(
    title='US Viewers Over Time (with Missing Values Filled from Second Dataset)'
)

season_finals = df2[df2['is_last_episode'] == 1].copy()
limit_bounds = alt.Chart(season_finals).mark_rule(color='black', strokeDash=[4, 4], opacity=0.6).encode(
    x=alt.X('original_air_date:T'),
    tooltip=[alt.Tooltip('season:N', title='Season')]
)
chart = line_chart + limit_bounds
chart


alt.LayerChart(...)

In [24]:
# Q1
alt.Chart(df).mark_line().encode(
    x='original_air_date',
    y='average(imdb_rating)'
)

alt.Chart(...)

In [25]:
# Q2
alt.Chart(df).mark_line().encode(
    x='original_air_date',
    y='average(us_viewers_in_millions)'
)

alt.Chart(...)

In [26]:
# Q3
alt.Chart(df).mark_circle().encode(
    x='us_viewers_in_millions',
    y='imdb_rating'
)

alt.Chart(...)

In [27]:
# Q4
# Day Rating Distribution
alt.Chart(df).mark_bar().encode(
    x='day_aired',
    y='average(imdb_rating)'
)

alt.Chart(...)